# E2: ablation of *resembles* edges

Reruns FLOCK and ULTRA on the drug → disease queries after removing all *resembles* edges from the graph.
Compare it with the E1 notebook, which uses the same folds and seeds on the full graph.

- **Kaggle:** select the **GPU T4** accelerator, turn **Internet on**, choose **Save Version**, then
  **Save & Run All**. Download the files from the Output tab.
- **Colab:** choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Keep the tab
  open. The last cell downloads a zip of the results.

Each fold is saved as it finishes, and no new fold starts after `TIME_BUDGET_HOURS`.

In [ ]:
import base64, io, json, os, shutil, subprocess, tarfile, time
from pathlib import Path

TIME_BUDGET_HOURS = 8
NOTEBOOK = "gpu_e2"
EXPERIMENT_COMMIT = "2caa3b8d13769401367bb910ce47eb49502b5e52"
UPSTREAM_COMMIT = "f35103d25a78bdf4075de5c673a51de4979aa4d7"
ULTRA_COMMIT = "427966ad8ed60420eef034063d44f3153addff90"


def running_on_colab():
    try:
        import google.colab
    except ImportError:
        return False
    return "KAGGLE_KERNEL_RUN_TYPE" not in os.environ


ON_COLAB = running_on_colab()
WORK = Path(os.environ.get("GPU_WORK", "/tmp/gpu_run"))
OUTPUT = Path(os.environ.get("GPU_OUTPUT", f"/content/{NOTEBOOK}_output" if ON_COLAB else "/kaggle/working"))
REPO = WORK / "flock"
ULTRA = WORK / "ULTRA"
PYTHON = Path(os.environ.get("GPU_PYTHON", WORK / "env/bin/python"))
DEADLINE = time.time() + TIME_BUDGET_HOURS * 3600
RUN_ENV = {
    **os.environ,
    "EXPERIMENT_COMMIT": EXPERIMENT_COMMIT,
    "CUDA_VISIBLE_DEVICES": "0",
    "TQDM_DISABLE": "1",
    "PYTHONPATH": str(REPO / "experiments"),
}
WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)


def sh(command, check=True, cwd=None):
    print("$", command, flush=True)
    bash_command = f"set -o pipefail; {command}"
    code = subprocess.run(bash_command, shell=True, executable="/bin/bash", cwd=cwd, env=RUN_ENV).returncode
    if code and check:
        raise RuntimeError(f"command failed with exit code {code}")
    if code:
        print(f"FAILED with exit code {code}; continuing", flush=True)
    return code == 0


def stage(module, arguments):
    sh(
        f"{PYTHON} -m {module} {arguments} --deadline {DEADLINE} 2>&1 | tee -a {OUTPUT}/{NOTEBOOK}.log",
        check=False,
        cwd=REPO / "experiments",
    )

In [ ]:
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
).stdout
print(gpu)
assert "T4" in gpu, "Select the GPU T4 accelerator in the notebook settings."
sh("nproc")

In [ ]:
if not REPO.exists():
    sh(f"git clone -q https://github.com/jw9730/flock {REPO}")
sh(f"git -C {REPO} checkout -q {UPSTREAM_COMMIT}")
if not ULTRA.exists():
    sh(f"git clone -q https://github.com/DeepGraphLearning/ULTRA {ULTRA}")
sh(f"git -C {ULTRA} checkout -q {ULTRA_COMMIT}")
BUNDLE = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as archive:
    archive.extractall(REPO)
sh(f"bash {REPO}/experiments/common/get_hetionet.sh")
print("code and data ready")

In [ ]:
sh("pip install -q uv")
sh(f"uv venv -q --python 3.12 {WORK / 'env'}")
sh(f"uv pip install -q --python {PYTHON} --index-url https://download.pytorch.org/whl/cu126 torch==2.8.0")
sh(f"uv pip install -q --python {PYTHON} -r {REPO}/experiments/requirements-gpu.txt")
sh(
    f"uv pip install -q --python {PYTHON} --only-binary :all: torch-scatter==2.1.2 "
    "-f https://data.pyg.org/whl/torch-2.8.0+cu126.html"
)
sh(f"uv pip install -q --python {PYTHON} --no-deps .", cwd=REPO / "graph-walker")
sh(f"{PYTHON} -c 'import torch, graph_walker; print(torch.__version__, torch.cuda.get_device_name(0))'", cwd=WORK)
sh(f"uv pip freeze --python {PYTHON} > {OUTPUT}/{NOTEBOOK}_freeze.txt")

In [ ]:
stage(
    "gpu.run_entity",
    f"--model ultra --ultra-repo {ULTRA} --direction tail --condition no_resembles --output-dir {OUTPUT}/runs",
)

In [ ]:
stage(
    "gpu.run_entity",
    f"--model flock --direction tail --condition no_resembles --seeds 0,1 --output-dir {OUTPUT}/runs",
)

In [ ]:
sh(f"{PYTHON} -m gpu.summarise_runs --runs {OUTPUT}/runs 2>&1 | tee {OUTPUT}/{NOTEBOOK}_summary.txt",
   check=False, cwd=REPO / "experiments")

In [ ]:
if ON_COLAB:
    from google.colab import files

    files.download(shutil.make_archive(f"/content/{NOTEBOOK}_results", "zip", OUTPUT))